# MJO 4: Create MJO Lag Plots

Via `mjo_lag_lat_lon.ncl`

In [4]:
# # getenv == os.environ
import os

CASENAME = "QBOi.EXP1.AMIP.001"
startdate = '19790101'
enddate = '19811231'

# /glade/u/home/bundy/mdtf/MDTF_3_main/MDTF-diagnostics.blocking_notebook/diagnostics/MJO_suite/MJO_driver.py
#DATADIR = "/glade/u/home/bundy/diag/mdtf/inputdata/model/QBOi.EXP1.AMIP.001/"
WORK_DIR = os.getcwd()
DATADIR = "/data/"

U200_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.U200.day.nc"
V200_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.V200.day.nc"
U850_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.U850.day.nc"
V850_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.V850.day.nc"
RLUT_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.FLUT.day.nc"
PR_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.PRECT.day.nc"

lev_coord = "lev"
lat_coord = "lat"
lon_coord = "lon"
time_coord = "time"
pr_var = "PRECT"
rlut_var = "FLUT"

u200_var = "U200"
v200_var = "V200"

wk_dir = WORK_DIR + "/model/"

# setup data directory, if does not already exist
#if not os.path.exists(DATADIR): os.makedirs(DATADIR)

In [5]:
#var_types = ["pr", "u850"]
var_types = ["prect", "u850"]
#file_dir = WORK_DIR + "/model/"
file_dir = WORK_DIR + DATADIR + "anomaly/" + CASENAME
filename_pr = file_dir  + ".prect.day.anom.nc"
filename_u850 = file_dir  + ".u850.day.anom.nc"

nameSeason = ["winter", "summer", "annual"]

In [8]:
# Indian Ocean base region
nameRegion = "IO"
latS_IO = -10
latN_IO = 5
lonL_IO = 75
lonR_IO = 100

# global subset
latS_globe = -30
latN_globe = 30

# latitude band (lag, lon)
latn = 10
lats = -10

# longitude band for (lag, lat)
lonl = 80
lonr = 100

# output
pltName = CASENAME + ".MJO.lag.lat.lon"
pltType = "ps" # alteratively, x11, ps, eps, pdf, png
pltDir = WORK_DIR + "/PS/" # output plot directory

### Fixed Lanczos Filter Weights

Create bandpass filter from weights

See: [Lanczos Filter Weights](https://www.ncl.ucar.edu/Applications/mjoclivar.shtml)

```
When needed, the weights for the suggested 20-100 day bandpass Lanczos filter are generated 'on-the-fly' using:

  ihp      = 2                             ; bpf=>band pass filter
  nWgt     = 201
  sigma    = 1.0                           ; Lanczos sigma
  fca      = 1./100.
  fcb      = 1./20.
  wgt      = filwgts_lanczos (nWgt, ihp, fca, fcb, sigma )

```

See `generate_lanczos_filter_weights.ncl` which generates `lanczos_filter_weights_output.txt`

In [10]:
import numpy as np
def lanczos_filter_weights(nwt=None, ihp=None, fca=None, fcb=None, nsigma=None):
    # nwt = scale indicating the total number of weights (must be an odd number, nwt >= 3).The more weights, the better the filter, but greater loss of data
    # ihp = scale indicating the low-pass filter
    # fca = scale indicating the cut-off frequency of the ideal high or low-pass filter (0 < fca < 0.5)
    # fcb = scale used only when band-pass filter is desired, second cut-off frequency (fca < fcb < 0.5)
    # scale indicating the power of sigma factor (nsigma >= 0), ngima = 1 is common
    # returns: a symmetrical set of weights
    ncl_output_weights = np.loadtxt("lanczos_filter_weights_output.txt", delimiter=",", skiprows=17)
    return ncl_output_weights 

In [11]:
## calculate one-dimensional filter weights (filwgts_lanczos) 

# create BandPass filter
#ihp = 2 # bpf->band pass filter, 2 = band-pass
#nWgt = 201 # number of weights (must be an odd number, the more weights, the better the filter, but the greater loss of data)
#sigma = 1 # lanczos sigma
#fca = 1/100 # indicating the cut-off frequency of the ideal high/low-pass filter (0.0 < fca < 0.5)
#fcb = 1/20 # scalar used when band-pass filter is desired. It is the second cut-off frequency  (fca < fcb < 0.5)
weights = lanczos_filter_weights()
weights

array([ 1.933179e-11, -1.602059e-05, -4.602891e-05, -8.413403e-05,
       -1.211932e-04, -1.459612e-04, -1.466584e-04, -1.127676e-04,
       -3.682710e-05,  8.402883e-05,  2.470150e-04,  4.433134e-04,
        6.584777e-04,  8.736432e-04,  1.067462e-03,  1.218581e-03,
        1.308390e-03,  1.323730e-03,  1.259200e-03,  1.118744e-03,
        9.162520e-04,  6.749944e-04,  4.258647e-04,  2.045088e-04,
        4.758891e-05, -1.146049e-05,  5.326822e-05,  2.562978e-04,
        5.978383e-04,  1.062210e-03,  1.617924e-03,  2.219547e-03,
        2.811319e-03,  3.332239e-03,  3.722208e-03,  3.928611e-03,
        3.912660e-03,  3.654768e-03,  3.158300e-03,  2.451133e-03,
        1.584706e-03,  6.304432e-04, -3.262688e-04, -1.194086e-03,
       -1.884996e-03, -2.323995e-03, -2.458149e-03, -2.264029e-03,
       -1.752608e-03, -9.709108e-04, -1.243057e-09,  1.050766e-03,
        2.052896e-03,  2.870605e-03,  3.374352e-03,  3.454755e-03,
        3.035499e-03,  2.083817e-03,  6.173067e-04, -1.293891e

### Preciptation
- Time indices cooresponding to the desired time window
- Reader user specified period

In [13]:
import xarray as xr
file_pr = xr.open_dataset(filename_pr)
file_pr

<xarray.Dataset> Size: 565MB
Dimensions:  (time: 2555, lat: 192, lon: 288)
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Data variables:
    date     (time) float64 20kB ...
    PRECT    (time, lat, lon) float32 565MB ...

In [14]:
pr = file_pr[var_types[0].upper()].sel(lat=slice(latS_globe, latN_globe)) # L89 (mjo_lag_lat_lon.ncl)
pr

<xarray.DataArray 'PRECT' (time: 2555, lat: 64, lon: 288)> Size: 188MB
[47093760 values with dtype=float32]
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 512B -29.69 -28.74 -27.8 -26.86 ... 27.8 28.74 29.69
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8

In [15]:
time_pr = pr["time"]
date_pr = pr['time'].values.tolist()
time_pr

<xarray.DataArray 'time' (time: 2555)> Size: 20kB
array([cftime.DatetimeNoLeap(1975, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 2, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 3, 0, 0, 0, 0, has_year_zero=True), ...,
       cftime.DatetimeNoLeap(1981, 12, 29, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1981, 12, 30, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1981, 12, 31, 0, 0, 0, 0, has_year_zero=True)],
      shape=(2555,), dtype=object)
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
Attributes:
    long_name:  time
    bounds:     time_bnds

In [16]:
wypr = pr["lat"].sel(lat=slice(latS_IO, latN_IO)) # L97 (mjo_lag_lat_lon.ncl)
wypr

<xarray.DataArray 'lat' (lat: 16)> Size: 128B
array([-9.895288, -8.95288 , -8.010471, -7.068063, -6.125654, -5.183246,
       -4.240838, -3.298429, -2.356021, -1.413613, -0.471204,  0.471204,
        1.413613,  2.356021,  3.298429,  4.240838])
Coordinates:
  * lat      (lat) float64 128B -9.895 -8.953 -8.01 -7.068 ... 2.356 3.298 4.241
Attributes:
    units:      degrees_north
    long_name:  latitude

In [17]:
wypr = np.cos(0.017459 * wypr)
wypr

<xarray.DataArray 'lat' (lat: 16)> Size: 128B
array([0.98511376, 0.98780871, 0.99023625, 0.99239572, 0.99428653,
       0.99590818, 0.99726023, 0.99834231, 0.99915413, 0.99969546,
       0.99996616, 0.99996616, 0.99969546, 0.99915413, 0.99834231,
       0.99726023])
Coordinates:
  * lat      (lat) float64 128B -9.895 -8.953 -8.01 -7.068 ... 2.356 3.298 4.241

In [18]:
twStrt = time_pr["time"].min().item()
twLast = time_pr["time"].max().item()
print(f"Time range from {twStrt} to {twLast}")

Time range from 1975-01-01 00:00:00 to 1981-12-31 00:00:00


### U850 Anomalies
- Time indices cooresponding to the desired time window
- Reader user specified period

In [19]:
file_u850 = xr.open_dataset(filename_u850)
file_u850

<xarray.Dataset> Size: 565MB
Dimensions:  (time: 2555, lat: 192, lon: 288)
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Data variables:
    date     (time) float64 20kB ...
    U850     (time, lat, lon) float32 565MB ...

In [20]:
u850 = file_u850[var_types[1].upper()].sel(lat=slice(latS_globe, latN_globe)) # L89 (mjo_lag_lat_lon.ncl)
u850

<xarray.DataArray 'U850' (time: 2555, lat: 64, lon: 288)> Size: 188MB
[47093760 values with dtype=float32]
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 512B -29.69 -28.74 -27.8 -26.86 ... 27.8 28.74 29.69
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8

In [21]:
time_u850 = u850["time"]
date_u850 = u850['time'].values.tolist()
time_u850

<xarray.DataArray 'time' (time: 2555)> Size: 20kB
array([cftime.DatetimeNoLeap(1975, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 2, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 3, 0, 0, 0, 0, has_year_zero=True), ...,
       cftime.DatetimeNoLeap(1981, 12, 29, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1981, 12, 30, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1981, 12, 31, 0, 0, 0, 0, has_year_zero=True)],
      shape=(2555,), dtype=object)
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
Attributes:
    long_name:  time
    bounds:     time_bnds

In [22]:
wyu850 = u850["lat"].sel(lat=slice(latS_IO, latN_IO)) # L131 (mjo_lag_lat_lon.ncl)
wyu850

<xarray.DataArray 'lat' (lat: 16)> Size: 128B
array([-9.895288, -8.95288 , -8.010471, -7.068063, -6.125654, -5.183246,
       -4.240838, -3.298429, -2.356021, -1.413613, -0.471204,  0.471204,
        1.413613,  2.356021,  3.298429,  4.240838])
Coordinates:
  * lat      (lat) float64 128B -9.895 -8.953 -8.01 -7.068 ... 2.356 3.298 4.241
Attributes:
    units:      degrees_north
    long_name:  latitude

In [23]:
wyu850 = np.cos(0.017459 * wyu850)
wyu850

<xarray.DataArray 'lat' (lat: 16)> Size: 128B
array([0.98511376, 0.98780871, 0.99023625, 0.99239572, 0.99428653,
       0.99590818, 0.99726023, 0.99834231, 0.99915413, 0.99969546,
       0.99996616, 0.99996616, 0.99969546, 0.99915413, 0.99834231,
       0.99726023])
Coordinates:
  * lat      (lat) float64 128B -9.895 -8.953 -8.01 -7.068 ... 2.356 3.298 4.241

In [24]:
# ensure dates agree
for i in range(len(date_pr)):
    try:
        date_pr[i] == date_u850[i]
    except Exception:
        print("Date mismatch: exit")    

In [26]:
# Create weighted area average of the base IO precipation series (time)
pr_sub = pr.sel(lat=slice(latS_IO, latN_IO), lon=slice(lonL_IO, lonR_IO)) # L150 (mjo_lag_lat_lon.ncl)
pr_weights = wypr.sel(lat=slice(latS_IO, latN_IO))
pr_weighted = pr_sub.weighted(pr_weights) # weighted average
PIO = pr_weighted.mean(dim=["lat", "lon"])
# remove overall trendline (dtrend)
poly_fit_pr = pr_sub.polyfit(dim="time", deg=1)
fit_pr = xr.polyval(pr_sub["time"], poly_fit_pr.polyfit_coefficients)
PIO = pr_sub - fit_pr
# apply filter for leftdim (wgt_runave_leftdim) 
#TODO: L152